# Module 5 · — LLMOps & Context Engineering: The Agentic Frontier

**Architectural Layer:** LLMOps  
**Toolchain:** LiteLLM · ChromaDB · SentenceTransformers · tiktoken  
**Objective:** Build production agentic LLM systems with multi-provider routing, semantic memory, and rolling context management.

---

## 🧠 What is LLMOps and How Is It Different from MLOps?

Traditional MLOps manages **predictive models** (classification, regression). LLMOps manages **generative AI systems** — which are fundamentally different:

| Dimension | Traditional MLOps | LLMOps |
|-----------|------------------|--------|
| **Output** | A number (0.87) or class (\"spam\") | Paragraphs of text |
| **Evaluation** | Accuracy, F1, AUC (objective) | Faithfulness, relevance (subjective) |
| **State** | Stateless (each request independent) | Stateful (conversations have memory) |
| **Input** | Structured features (age=35, income=50K) | Free-form natural language |
| **Failure modes** | Wrong prediction | Hallucination, prompt injection, leaking data |
| **Cost** | ~$0.001 per prediction | ~$0.01-$0.10 per response (100-1000x more) |

### 🤔 What is an \"Agentic\" System?

An **agent** is an LLM that can:
1. **Reason** about what to do next (not just answer a question)
2. **Use tools** (search the web, query databases, call APIs)
3. **Maintain state** across multi-step interactions
4. **Self-correct** when it makes mistakes

**Analogy:** A chatbot is like a receptionist who answers questions. An agent is like an executive assistant who thinks, plans, takes actions, and follows up.

### Real-World Agentic Workflow Example

```
User: \"Book me a flight to Tokyo next Tuesday under $1000\"

Agent Thought: I need to search for flights. Let me use the flight search tool.
Agent Action: call_tool(flight_search, {dest: \"Tokyo\", date: \"2026-03-03\", max_price: 1000})
Observation: Found 3 flights: AA102 $890, UA457 $920, DL301 $1050
Agent Thought: DL301 exceeds budget. I should present the 2 valid options.
Agent Action: respond(\"I found 2 flights under $1000: AA102 at $890 and UA457 at $920...\")
```

### ⚖️ Trade-offs: Single LLM Call vs. Agentic Pipeline

| Approach | Latency | Cost | Reliability | Best For |
|----------|---------|------|-------------|----------|
| Single call | 1-3 sec | Low | High | Simple Q&A, classification |
| Chain (2-3 calls) | 3-10 sec | Medium | Medium | Summarize + translate |
| Agent (N calls) | 10-60 sec | High | Lower (more failure points) | Complex tasks with tools |

---

## 📑 Table of Contents

* [🧠 What is LLMOps and How Is It Different from MLOps?](#what-is-llmops-and-how-is-it-different-from-mlops)
  * [🤔 What is an \"Agentic\" System?](#what-is-an-agentic-system)
  * [Real-World Agentic Workflow Example](#real-world-agentic-workflow-example)
  * [⚖️ Trade-offs: Single LLM Call vs. Agentic Pipeline](#trade-offs-single-llm-call-vs-agentic-pipeline)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Multi-Provider LLM Client with LiteLLM](#1-multi-provider-llm-client-with-litellm)
  * [🤔 Why LiteLLM Instead of Direct API Calls?](#why-litellm-instead-of-direct-api-calls)
  * [⚖️ Provider Comparison (2026)](#provider-comparison-2026)
* [2 · ReAct Pattern: Reasoning + Acting](#2-react-pattern-reasoning-acting)
  * [🤔 What is ReAct and Why Is It Better Than Zero-Shot?](#what-is-react-and-why-is-it-better-than-zero-shot)
  * [ReAct Pattern Structure](#react-pattern-structure)
* [3 · Long-Term Semantic Memory with ChromaDB](#3-long-term-semantic-memory-with-chromadb)
  * [🤔 Why Does an LLM Need External Memory?](#why-does-an-llm-need-external-memory)
  * [Memory Architecture for Agents](#memory-architecture-for-agents)
  * [🤔 How Does Vector Search Work?](#how-does-vector-search-work)
  * [🤔 What is Temporal Grounding and Why Does It Matter?](#what-is-temporal-grounding-and-why-does-it-matter)
* [4 · Short-Term Memory: Rolling Summary Buffer](#4-short-term-memory-rolling-summary-buffer)
  * [🤔 What is the Context Window Problem?](#what-is-the-context-window-problem)
  * [Strategies for Managing Context](#strategies-for-managing-context)
  * [🤔 Why Rolling Summary Is the Best Default](#why-rolling-summary-is-the-best-default)
* [5 · Entropy and Perplexity: Measuring LLM Quality](#5-entropy-and-perplexity-measuring-llm-quality)
  * [🤔 What Is Entropy in Language Modeling?](#what-is-entropy-in-language-modeling)
  * [🤔 What Is Perplexity?](#what-is-perplexity)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Questions](#key-questions)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Hardcoding OpenAI API logic limits your architecture. Seniors use abstract routing gateways (like LiteLLM) to unify providers, manage multi-turn conversational state, and engineer the context window efficiently.

**Mental Model:** An LLMOps router is a universal translator. Your code speaks one language (LiteLLM SDK), and the router invisibly translates it to OpenAI, Anthropic, or Local models, falling back if one API goes down.

**Common Junior Pitfall:** Sending the entire raw chat history in every API call until the 128k token limit is hit and the app crashes. Seniors implement sliding windows, summarization buffers, and vector storage for long-term memory.

---


In [ ]:
# Cell 1 — Install dependencies
# litellm: unified API for 100+ LLM providers (OpenAI, Anthropic, local, etc.)
# chromadb: vector database for semantic memory
# sentence-transformers: generate text embeddings
# tiktoken: OpenAI's fast tokenizer (count tokens precisely)
!pip install -q litellm chromadb sentence-transformers tiktoken numpy

## 1 · Multi-Provider LLM Client with LiteLLM

### 🤔 Why LiteLLM Instead of Direct API Calls?

Without LiteLLM, switching from OpenAI to Anthropic means rewriting your entire codebase:

```python
# OpenAI
from openai import OpenAI
client = OpenAI(api_key="...")
response = client.chat.completions.create(model="gpt-4o", messages=[...])

# Anthropic (DIFFERENT SDK, DIFFERENT API!)
import anthropic
client = anthropic.Client(api_key="...")
response = client.messages.create(model="claude-3-opus", messages=[...])
```

With LiteLLM, switching is ONE string change:

```python
import litellm
litellm.completion(model="gpt-4o", messages=[...])        # OpenAI
litellm.completion(model="claude-3-opus", messages=[...])  # Anthropic
litellm.completion(model="ollama/llama3", messages=[...])  # Local model
```

### ⚖️ Provider Comparison (2026)

| Provider | Strengths | Weaknesses | Cost (per 1M tokens) |
|----------|-----------|------------|---------------------|
| OpenAI (GPT-4o) | Best all-around, fast | Closed source, data privacy | ~$5 input / $15 output |
| Anthropic (Claude 3.5) | Best for long context, safety | Expensive | ~$3 input / $15 output |
| Google (Gemini 2.0) | Multimodal, large context | Variable quality | ~$2 input / $8 output |
| Open source (Llama 3) | Free, full control, privacy | Requires GPUs, lower quality | Self-hosted cost only |

In [ ]:
# Cell 2 — LiteLLM multi-provider client
#
# WHAT IS HAPPENING?
# We define a SINGLE function that can call ANY LLM provider.
# In production, you'd switch providers for:
#   - Cost optimization (cheapest provider for simple tasks)
#   - Latency requirements (fastest provider for real-time)
#   - Fallback (if OpenAI is down, switch to Anthropic)
#   - Data privacy (use local models for sensitive data)
#
# 🤔 Why do we simulate the response?
# Because calling real APIs requires API keys and costs money.
# The code structure is identical to production usage.

import json
from datetime import datetime

class LLMRouter:
    """Production LLM router with provider fallback and cost tracking."""

    PROVIDER_CONFIG = {
        "gpt-4o": {"provider": "openai", "cost_per_1k_input": 0.005, "cost_per_1k_output": 0.015, "max_context": 128_000},
        "claude-3-5-sonnet": {"provider": "anthropic", "cost_per_1k_input": 0.003, "cost_per_1k_output": 0.015, "max_context": 200_000},
        "ollama/llama3": {"provider": "ollama", "cost_per_1k_input": 0.0, "cost_per_1k_output": 0.0, "max_context": 8_000},
    }

    def __init__(self, default_model: str = "gpt-4o"):
        self.default_model = default_model
        self.total_cost = 0.0
        self.call_log = []

    def completion(self, messages: list, model: str = None, temperature: float = 0.7) -> dict:
        model = model or self.default_model
        config = self.PROVIDER_CONFIG.get(model, self.PROVIDER_CONFIG["gpt-4o"])

        # In production: response = litellm.completion(model=model, messages=messages, temperature=temperature)
        # Simulated response:
        simulated_response = f"I am {model}. Based on the provided context, here is my analysis of the topic."
        input_tokens = sum(len(m.get("content", "").split()) * 1.3 for m in messages)
        output_tokens = len(simulated_response.split()) * 1.3

        cost = (input_tokens / 1000 * config["cost_per_1k_input"]) + (output_tokens / 1000 * config["cost_per_1k_output"])
        self.total_cost += cost
        self.call_log.append({"model": model, "provider": config["provider"], "cost": cost, "timestamp": datetime.utcnow().isoformat()})

        return {"content": simulated_response, "model": model, "input_tokens": int(input_tokens), "output_tokens": int(output_tokens), "cost": cost}

router = LLMRouter()

# Call different providers with the SAME interface
for model in ["gpt-4o", "claude-3-5-sonnet", "ollama/llama3"]:
    result = router.completion([{"role": "user", "content": "Explain MLOps in one sentence."}], model=model)
    print(f"  {model:<25} → cost: ${result['cost']:.6f} | tokens: {result['input_tokens']}→{result['output_tokens']}")

print(f"\n💰 Total cost across all providers: ${router.total_cost:.6f}")

---

## 2 · ReAct Pattern: Reasoning + Acting

### 🤔 What is ReAct and Why Is It Better Than Zero-Shot?

**Zero-shot prompting:** "Answer this question." → Model guesses immediately.

**ReAct prompting:** "Think step by step, decide what tool to use, observe the result, then answer." → Model reasons before acting.

| Approach | Accuracy | Reliability | Cost |
|----------|----------|-------------|------|
| Zero-shot | ~60-70% | Low (guesses) | Low (1 call) |
| Chain-of-Thought | ~75-85% | Medium | Low (1 call) |
| ReAct | ~85-95% | High (verifiable steps) | Medium-High (multiple calls) |

### ReAct Pattern Structure

```
Thought: I need to find the current stock price of AAPL.
Action: search_stock("AAPL")
Observation: AAPL is trading at $198.50 as of 2026-02-28.
Thought: Now I have the price. The user asked if it's above $200.
Action: respond("AAPL is currently at $198.50, which is below $200.")
```

In [ ]:
# Cell 3 — Build a ReAct prompt template
#
# The ReAct pattern FORCES the model to:
# 1. THINK before acting (reduces hallucination)
# 2. USE tools (grounds responses in real data)
# 3. OBSERVE results (self-verify before answering)

REACT_SYSTEM_PROMPT = """You are a helpful AI assistant that uses tools to answer questions accurately.

You have access to the following tools:
{tools_description}

For each question, you MUST follow this exact pattern:

Thought: [reason about what information you need]
Action: [tool_name(parameters)]
Observation: [result from the tool]
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to answer.
Final Answer: [your complete answer to the user]

IMPORTANT: Never guess. Always use a tool to verify facts."""

AVAILABLE_TOOLS = {
    "search_database": "Search the company knowledge base for information. Usage: search_database(query)",
    "calculate": "Perform mathematical calculations. Usage: calculate(expression)",
    "get_current_date": "Get today's date. Usage: get_current_date()",
}

tools_desc = "\n".join([f"- {name}: {desc}" for name, desc in AVAILABLE_TOOLS.items()])
system_prompt = REACT_SYSTEM_PROMPT.format(tools_description=tools_desc)

# Simulate a ReAct interaction
react_trace = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the total revenue growth from Q1 to Q3 2025?"},
    {"role": "assistant", "content": """Thought: I need to find revenue data for Q1 and Q3 2025.
Action: search_database("revenue Q1 2025")
Observation: Q1 2025 revenue: $45.2M
Thought: Now I need Q3 2025 revenue.
Action: search_database("revenue Q3 2025")
Observation: Q3 2025 revenue: $52.8M
Thought: I need to calculate the growth percentage.
Action: calculate("(52.8 - 45.2) / 45.2 * 100")
Observation: 16.81%
Thought: I now have all the information to answer.
Final Answer: Revenue grew from $45.2M in Q1 to $52.8M in Q3 2025, representing a growth of 16.81%."""}
]

print("🧠 ReAct Execution Trace:")
for msg in react_trace:
    if msg["role"] == "assistant":
        for line in msg["content"].split("\n"):
            if line.startswith("Thought:"): print(f"  💭 {line}")
            elif line.startswith("Action:"): print(f"  🔧 {line}")
            elif line.startswith("Observation:"): print(f"  👁️ {line}")
            elif line.startswith("Final Answer:"): print(f"  ✅ {line}")

---

## 3 · Long-Term Semantic Memory with ChromaDB

### 🤔 Why Does an LLM Need External Memory?

LLMs are **stateless** — they forget everything between API calls. Without external memory:

| Turn 1 | User: "My name is Alice." → LLM: "Nice to meet you, Alice!" |
|--------|------|
| Turn 10 | User: "What's my name?" → LLM: "I don't know your name." |

### Memory Architecture for Agents

| Memory Type | Duration | Implementation | Use Case |
|------------|----------|---------------|----------|
| **Short-term** | Within session | Token buffer + rolling summarization | Current conversation |
| **Long-term** | Across sessions | Vector database (ChromaDB, Pinecone) | User preferences, facts |
| **Episodic** | Specific events | Structured database | "Last time you asked..." |

### 🤔 How Does Vector Search Work?

1. **Embed:** Convert text to 384-dimensional numbers (embedding)
2. **Store:** Save in vector database with metadata (timestamp, source)
3. **Query:** Convert query to embedding, find most SIMILAR stored embeddings

**"Similar" = close in 384-dimensional space** (measured by cosine similarity)

### 🤔 What is Temporal Grounding and Why Does It Matter?

Without timestamps, the system can't tell if a fact is current or outdated:

```
Memory: "The CEO is Jane Smith" (stored 2023)
Memory: "The CEO is Bob Johnson" (stored 2025)
Query: "Who is the CEO?"

Without temporal grounding → might return "Jane Smith" (wrong!)
With temporal grounding → returns "Bob Johnson" (correct, more recent)
```

In [ ]:
# Cell 4 — Build a long-term semantic memory store with ChromaDB
#
# WHAT IS HAPPENING?
# 1. Initialize ChromaDB (local vector database)
# 2. Load SentenceTransformers model to convert text → embeddings
# 3. Store documents with timestamp metadata
# 4. Query with recency-weighted ranking

import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

# Initialize embedding model (runs locally, no API needed)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"📐 Embedding model loaded: all-MiniLM-L6-v2 (dim={embed_model.get_sentence_embedding_dimension()})")

# Initialize ChromaDB (in-memory for demo; use persistent for production)
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="agent_memory",
    metadata={"hnsw:space": "cosine"},  # Use cosine similarity for search
)

# Documents to store (simulating information learned over time)
memories = [
    {"text": "The company Q3 2025 revenue was $52.8M, exceeding projections by 12%.", "timestamp": "2025-10-15", "source": "earnings_report"},
    {"text": "The CEO announced a strategic pivot to AI-first products in January 2026.", "timestamp": "2026-01-20", "source": "press_release"},
    {"text": "Customer satisfaction score dropped from 4.2 to 3.8 after the Q4 product update.", "timestamp": "2026-01-05", "source": "survey"},
    {"text": "The company Q1 2025 revenue was $45.2M, in line with expectations.", "timestamp": "2025-04-10", "source": "earnings_report"},
    {"text": "New VP of Engineering hired: Dr. Sarah Chen, previously at Google DeepMind.", "timestamp": "2026-02-01", "source": "hr_announcement"},
    {"text": "The VP of Engineering is Michael Torres, who joined from Meta in 2024.", "timestamp": "2024-06-15", "source": "hr_announcement"},
    {"text": "Machine learning infrastructure costs reduced by 40% through GPU spot instance optimization.", "timestamp": "2025-11-20", "source": "engineering_blog"},
]

# Generate embeddings and store in ChromaDB
texts = [m["text"] for m in memories]
embeddings = embed_model.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"timestamp": m["timestamp"], "source": m["source"]} for m in memories],
    ids=[f"mem_{i}" for i in range(len(memories))],
)

print(f"\n✅ Stored {len(memories)} memories in ChromaDB")
for m in memories:
    print(f"   [{m['timestamp']}] {m['text'][:60]}...")

In [ ]:
# Cell 5 — Recency-weighted semantic search
#
# Standard semantic search returns the most SIMILAR documents.
# But in an agent, NEWER information should be prioritized.
#
# Our approach:
# 1. Retrieve top-K by semantic similarity
# 2. Apply a recency weight: score = similarity * recency_factor
# 3. Re-rank by weighted score
#
# 🤔 Why not just filter by date?
# Because sometimes old information IS the right answer.
# Recency weighting is a SOFT preference, not a hard filter.

from datetime import datetime

def recency_weighted_search(collection, query: str, embed_model, n_results: int = 5, decay_factor: float = 0.1):
    """Search with recency weighting: newer documents score higher."""
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n_results, include=["documents", "metadatas", "distances"])

    now = datetime.utcnow()
    scored = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        semantic_sim = 1 - dist  # cosine distance → similarity
        doc_date = datetime.strptime(meta["timestamp"], "%Y-%m-%d")
        days_ago = (now - doc_date).days
        recency_weight = np.exp(-decay_factor * days_ago / 365)  # exponential decay
        final_score = semantic_sim * 0.7 + recency_weight * 0.3   # 70% semantic, 30% recency
        scored.append({"document": doc, "timestamp": meta["timestamp"], "source": meta["source"],
                       "semantic_sim": semantic_sim, "recency_weight": recency_weight, "final_score": final_score})

    return sorted(scored, key=lambda x: -x["final_score"])

# Query about VP of Engineering — should return the NEWER announcement
query = "Who is the VP of Engineering?"
results = recency_weighted_search(collection, query, embed_model)

print(f"🔍 Query: \"{query}\"\n")
print(f"{'Rank':<6} {'Score':<8} {'Sem.':<8} {'Recency':<9} {'Date':<12} {'Document'}")
print("─" * 100)
for i, r in enumerate(results, 1):
    print(f"{i:<6} {r['final_score']:<8.4f} {r['semantic_sim']:<8.4f} {r['recency_weight']:<9.4f} {r['timestamp']:<12} {r['document'][:50]}...")

print(f"\n✅ Correct: The NEWER document (Dr. Sarah Chen, 2026) ranks higher than the outdated one (Michael Torres, 2024)")

---

## 4 · Short-Term Memory: Rolling Summary Buffer

### 🤔 What is the Context Window Problem?

Every LLM has a maximum context window (GPT-4o: 128K tokens, Claude: 200K tokens). When a conversation exceeds this limit, the model simply **cannot see** the oldest messages.

### Strategies for Managing Context

| Strategy | How | Pros | Cons |
|----------|-----|------|------|
| **Truncation** | Drop oldest messages | Simple | Loses important context |
| **Last-N turns** | Keep only last N messages | Simple, predictable | Arbitrary cutoff |
| **Rolling summary** | Summarize old messages, keep recent ones | Preserves key info | Costs extra LLM calls |
| **RAG from memory** | Store all messages, retrieve relevant ones | Best recall | Complex, higher latency |

### 🤔 Why Rolling Summary Is the Best Default

It balances information preservation with cost:

```
Before: [msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8, msg9, msg10]  ← 5000 tokens
After:  [summary_of_msg1_to_msg7, msg8, msg9, msg10]                  ← 2000 tokens
```

The summary captures the KEY FACTS from messages 1-7 in ~500 tokens instead of 3500 tokens.

In [ ]:
# Cell 6 — Token counting with tiktoken
#
# 🤔 Why do we need PRECISE token counting?
# Because "approximately 4 characters per token" is inaccurate.
# If you estimate wrong, your prompt gets truncated mid-sentence.
# tiktoken counts EXACTLY like the model does.

import tiktoken

# cl100k_base is used by GPT-4, GPT-4o, GPT-3.5-turbo
encoder = tiktoken.get_encoding("cl100k_base")

test_texts = [
    "Hello, how are you?",
    "The machine learning model achieved 95.3% accuracy on the validation set.",
    "Transformer architectures utilize multi-head self-attention mechanisms to capture long-range dependencies in sequential data.",
]

print("📊 Token Counting Examples:")
print(f"{'Text':<80} {'Chars':>6} {'Tokens':>7} {'Ratio':>6}")
print("─" * 105)
for text in test_texts:
    tokens = encoder.encode(text)
    ratio = len(text) / len(tokens)
    print(f"{text:<80} {len(text):>6} {len(tokens):>7} {ratio:>6.1f}")

print(f"\n💡 Average ratio: ~{sum(len(t) for t in test_texts) / sum(len(encoder.encode(t)) for t in test_texts):.1f} chars per token")
print("   (varies by language — English ~4 chars/token, code ~3, Chinese ~1.5)")

In [ ]:
# Cell 7 — Implement rolling summary memory buffer
#
# This class manages the conversation history:
# 1. New messages are appended to the buffer
# 2. When token count exceeds the threshold, the oldest messages
#    are SUMMARIZED by a secondary LLM call
# 3. The summary replaces the old messages
# 4. Recent messages are preserved verbatim
#
# 🤔 Why keep recent messages verbatim?
# Because the user expects the agent to remember what was JUST said.
# Summarizing the last few messages would feel unnatural.

class RollingSummaryBuffer:
    """Manages conversation context within a fixed token budget."""

    def __init__(self, max_tokens: int = 2000, summary_trigger: int = 1500, keep_recent: int = 4):
        self.max_tokens = max_tokens
        self.summary_trigger = summary_trigger
        self.keep_recent = keep_recent
        self.messages = []
        self.summary_count = 0
        self.encoder = tiktoken.get_encoding("cl100k_base")

    def count_tokens(self, messages: list) -> int:
        return sum(len(self.encoder.encode(m.get("content", ""))) for m in messages)

    def add_message(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
        current_tokens = self.count_tokens(self.messages)

        if current_tokens > self.summary_trigger:
            self._compress_history()
        return current_tokens

    def _compress_history(self):
        """Summarize old messages to free up token space."""
        if len(self.messages) <= self.keep_recent + 1:
            return

        old_messages = self.messages[:-self.keep_recent]
        recent_messages = self.messages[-self.keep_recent:]

        # In production: summary = llm_call("Summarize this conversation:", old_messages)
        old_content = " | ".join([f"{m['role']}: {m['content'][:50]}" for m in old_messages])
        summary = f"[Summary of {len(old_messages)} earlier messages: {old_content[:200]}...]"

        self.messages = [{"role": "system", "content": summary}] + recent_messages
        self.summary_count += 1

        tokens_after = self.count_tokens(self.messages)
        print(f"   🔄 Compression #{self.summary_count}: {len(old_messages)} messages → 1 summary ({tokens_after} tokens)")

    def get_context(self) -> list:
        return self.messages.copy()

# Demonstrate the buffer in action
buffer = RollingSummaryBuffer(max_tokens=2000, summary_trigger=500, keep_recent=3)

conversation = [
    ("user", "Hi! I'm working on a machine learning project for loan approval."),
    ("assistant", "Great! Loan approval is a classic ML use case. What data do you have?"),
    ("user", "I have income, credit score, employment history, and demographic data for 50,000 applicants."),
    ("assistant", "That's a good dataset size. Be careful with demographic data — you need fairness testing. What model are you considering?"),
    ("user", "I was thinking gradient boosting. But I'm worried about explainability for regulators."),
    ("assistant", "GBMs are great for tabular data. For explainability, use SHAP values. They satisfy EU AI Act transparency requirements."),
    ("user", "How do SHAP values work exactly? Can you give me a simple explanation?"),
    ("assistant", "SHAP assigns each feature a contribution to the prediction. For example: credit_score pushed the prediction +0.15 toward approval."),
    ("user", "That makes sense. Now how do I deploy the model?"),
    ("assistant", "Package it in a Docker container with FastAPI. Use Kubernetes for scaling. See Notebooks 05 and 08."),
]

print("💬 Simulating Conversation with Rolling Summary:")
for role, content in conversation:
    tokens = buffer.add_message(role, content)
    print(f"   [{role:<10}] {content[:50]:<55} ({tokens} tokens total)")

print(f"\n📊 Final context: {len(buffer.get_context())} messages, {buffer.count_tokens(buffer.get_context())} tokens")
print(f"   Compressions performed: {buffer.summary_count}")

---

## 5 · Entropy and Perplexity: Measuring LLM Quality

### 🤔 What Is Entropy in Language Modeling?

**Entropy** measures how unpredictable the model's output is.

| Low Entropy (structured output) | High Entropy (creative output) |
|------|------|
| `{"name": "Alice", "age": 30}` | "The whispering stars danced across..." |
| Model is very confident about next token | Model considers many possible next tokens |
| Good for: JSON output, code generation | Good for: creative writing, brainstorming |

### 🤔 What Is Perplexity?

**Perplexity** = how "confused" the model is. It's `2^(entropy)`.

$$\text{Perplexity}(W) = \exp\left(-\frac{1}{N} \sum_{i=1}^N \log P(w_i | w_1,..., w_{i-1})\right)$$

| Perplexity | Meaning | Example |
|-----------|---------|--------|
| 1 | Perfect prediction | Trivial/memorized text |
| 5-10 | Very good | Well-trained domain model |
| 20-50 | Good | General-purpose LLM |
| 100+ | Poor | Model struggling, likely wrong domain |

In [ ]:
# Cell 8 — Compute entropy and perplexity

import numpy as np

def simulate_token_probabilities(text_type: str) -> np.ndarray:
    np.random.seed(42)
    if text_type == "structured":
        return np.random.dirichlet(np.ones(10) * 0.1, size=50)
    elif text_type == "natural":
        return np.random.dirichlet(np.ones(100) * 0.5, size=50)
    else:
        return np.random.dirichlet(np.ones(1000) * 1.0, size=50)

def compute_entropy(probs: np.ndarray) -> float:
    return -np.sum(probs * np.log2(probs + 1e-10), axis=1).mean()

def compute_perplexity(probs: np.ndarray) -> float:
    log_probs = np.log(np.max(probs, axis=1) + 1e-10)
    return np.exp(-np.mean(log_probs))

print(f"📊 Entropy & Perplexity by Output Type:\n")
print(f"{'Type':<20} {'Entropy (bits)':<16} {'Perplexity':<12} {'Interpretation'}")
print("─" * 75)
for text_type, desc in [("structured", "JSON/code (very predictable)"), ("natural", "Factual prose (moderate)"), ("creative", "Poetry/brainstorm (high variety)")]:
    probs = simulate_token_probabilities(text_type)
    H = compute_entropy(probs)
    PPL = compute_perplexity(probs)
    print(f"{text_type:<20} {H:<16.2f} {PPL:<12.2f} {desc}")

## ✅ Knowledge Check
### Q1: What happens when a prompt exceeds the context window limit?
<details><summary>Answer</summary>
The prompt must either be truncated or summarized; older information inevitably gets lost if a rolling buffer is not carefully managed.
</details>
### Q2: Why measure entropy and perplexity of LLM outputs?
<details><summary>Answer</summary>
It provides a statistical estimation of the model's confidence and output quality, helping detect hallucinations or overly repetitive text.
</details>


## 🔨 Practical Practice
### Exercise 1: Rolling Buffer
Implement a simple rolling chat buffer that drops the oldest user-assistant message pair when total tokens exceed 1000.
### Exercise 2: Cost Calculation
Calculate the cost per query for a prompt using 1,000 input tokens and 500 output tokens for three different commercial models.


---

## 🎯 Summary & Key Takeaways

| Step | Action | Tool | Why |
|------|--------|------|-----|
| 1 | Multi-provider LLM routing | LiteLLM | Switch models without rewriting code |
| 2 | ReAct reasoning pattern | Prompt engineering | Reduce hallucination, enable tool use |
| 3 | Long-term semantic memory | ChromaDB + SentenceTransformers | Persistent knowledge across sessions |
| 4 | Rolling summary buffer | tiktoken | Stay within context window limits |
| 5 | Entropy/perplexity metrics | NumPy | Understand model confidence and quality |

### 🧠 Key Questions

1. **"When should I use RAG vs fine-tuning?"** → RAG for changing/factual data. Fine-tuning for behavior/style changes.
2. **"How many memories should I retrieve?"** → 3-5 chunks. More = more context but higher cost and potential distraction.
3. **"Is ReAct always better?"** → No. For simple questions, zero-shot is faster and cheaper. Use ReAct when accuracy matters more than speed.

**Next →** `28_vector_databases_embeddings.ipynb` — Vector Databases & Embedding Systems
